# Results

Every plot from the Results_Viewer notebooks, drawn with `examples/incremental_decision_tree/results_viewer.py`.
Run from the repo root.

In [ ]:
import sys
sys.path.insert(0, "examples/incremental_decision_tree")

import pandas as pd
from IPython.display import display

import results_viewer as rv

In [ ]:
run_id = "261117_152629"
smc_results, mcmc_results = rv.load_results(run_id)

# DA, Flat HINTS, HINTS, MH, largest subsample first: plot_metric_split's order.
smc_combined = rv.combine_results(smc_results)
mcmc_combined = rv.combine_results(mcmc_results)

# Runtime and ESS (SMC)

In [ ]:
rv.plot_runtime_per_iter(rv.by_proposal(smc_results), "SMC",ncols=4)
rv.plot_runtime_per_iter(rv.by_proposal(mcmc_results), "MCMC",ncols=4)

In [ ]:
rv.plot_ess(rv.by_proposal(smc_results),ncols=4)
rv.plot_ess(rv.by_proposal(smc_results), clip_to="HINTS",ncols=4)

# SMC results

In [ ]:
footnote = rv.mh_footnote(smc_combined, "step")
footnote

In [ ]:
rv.plot_chains(rv.by_proposal(smc_results), "tree_len", "SMC", "Tree Length", x="iteration", footnote=footnote,ncols=2)
rv.plot_chains(rv.by_proposal(smc_results), "tree_len", "SMC", "Tree Length", x="time", footnote=footnote,ncols=2)

In [ ]:
rv.plot_chains(rv.by_proposal(smc_results), "train_accuracy", "SMC", "Training Accuracy", x="iteration", footnote=footnote,ncols=4)
rv.plot_chains(rv.by_proposal(smc_results), "train_accuracy", "SMC", "Training Accuracy", x="time", footnote=footnote,ncols=4)

In [ ]:
rv.plot_metric_split(smc_combined, 'tree_len', "Tree Size", "SMC", "Internal nodes", footnote=footnote)
rv.plot_metric_split(smc_combined, 'tree_len', "Tree Size", "SMC", "Internal nodes", x="time", footnote=footnote)

In [ ]:
rv.plot_metric_split(smc_combined, 'train_accuracy', "Training Accuracy", "SMC", "Accuracy", footnote=footnote)
rv.plot_metric_split(smc_combined, 'train_accuracy', "Training Accuracy", "SMC", "Accuracy", x="time", footnote=footnote)

In [ ]:
rv.plot_metric_split(smc_combined, 'test_accuracy', "Test Accuracy", "SMC", "Accuracy", footnote=footnote)
rv.plot_metric_split(smc_combined, 'test_accuracy', "Test Accuracy", "SMC", "Accuracy", x="time", footnote=footnote)

In [ ]:
rv.plot_metric_split(smc_combined, 'test_brier', "Test Brier Score", "SMC", "Brier Score", footnote=footnote)
rv.plot_metric_split(smc_combined, 'test_brier', "Test Brier Score", "SMC", "Brier Score", x="time", footnote=footnote)

In [ ]:
rv.plot_metric_split(smc_combined, 'test_macro_f1', "Test Macro F1", "SMC", "Macro F1 Score", footnote=footnote)
rv.plot_metric_split(smc_combined, 'test_macro_f1', "Test Macro F1", "SMC", "Macro F1 Score", x="time", footnote=footnote)

In [ ]:
rv.plot_metric_split(smc_combined, 'test_log_loss', "Test Log Loss", "SMC", "Log Loss", footnote=footnote)
rv.plot_metric_split(smc_combined, 'test_log_loss', "Test Log Loss", "SMC", "Log Loss", x="time", footnote=footnote)

# MCMC results

In [ ]:
footnote = rv.mh_footnote(mcmc_combined, "iteration")
footnote

In [ ]:
rv.plot_chains(mcmc_results, "tree_len", "MCMC", "Tree Size", x="iteration", footnote=footnote)

In [ ]:
rv.plot_chains(mcmc_results, "test_accuracy", "MCMC", "Test Accuracy", x="iteration", footnote=footnote)

In [ ]:
rv.plot_metric_split(mcmc_combined, 'tree_len', "Tree Size", "MCMC", "Internal nodes", footnote=footnote)
rv.plot_metric_split(mcmc_combined, 'tree_len', "Tree Size", "MCMC", "Internal nodes", x="time", footnote=footnote)

In [ ]:
rv.plot_metric_split(mcmc_combined, 'train_accuracy', "Training Accuracy", "MCMC", "Accuracy", footnote=footnote)
rv.plot_metric_split(mcmc_combined, 'train_accuracy', "Training Accuracy", "MCMC", "Accuracy", x="time", footnote=footnote)

In [ ]:
rv.plot_metric_split(mcmc_combined, 'test_accuracy', "Test Accuracy", "MCMC", "Accuracy", footnote=footnote)
rv.plot_metric_split(mcmc_combined, 'test_accuracy', "Test Accuracy", "MCMC", "Accuracy", x="time", footnote=footnote)

In [ ]:
rv.plot_metric_split(mcmc_combined, 'test_log_loss', "Test Log Loss", "MCMC", "Log Loss", footnote=footnote)
rv.plot_metric_split(mcmc_combined, 'test_log_loss', "Test Log Loss", "MCMC", "Log Loss", x="time", footnote=footnote)

# Move outcomes and fates
What the proposal did with each move, then what became of it once the subsample screen is crossed with the sampler's own accept/reject.

In [ ]:
# Proposals alphabetically, MH last.
smc_moves = rv.mh_last(smc_combined)
mcmc_moves = rv.mh_last(mcmc_combined)

rv.plot_move_outcomes(smc_moves, "SMC", share=False)
rv.plot_move_outcomes(mcmc_moves, "MCMC")

In [ ]:
rv.plot_move_fates(mcmc_moves, "MCMC")
rv.plot_move_fates(mcmc_moves, "MCMC", grouped=True)
# SMC reweights particles instead of rejecting them, so the Reject fates are
# empty there and Accept means the proposal survived.
rv.plot_move_fates(smc_moves, "SMC", footnote=rv.SMC_FATE_NOTE)
rv.plot_move_fates(smc_moves, "SMC", grouped=True, footnote=rv.SMC_FATE_NOTE_GROUPED)

In [ ]:
mcmc_counts = rv.move_fate_table(mcmc_moves, "MCMC", grouped=True)
smc_counts = rv.move_fate_table(smc_moves, "SMC", grouped=True)
display(pd.DataFrame(mcmc_counts))

In [ ]:
print(rv.create_latex_table(mcmc_counts, "MCMC"))

## Acceptance by move type at the tree-size cap
How often each move type is accepted (MCMC) or kept (SMC) when the tree it was proposed on is at `max_tree_size`, compared with below it. At the cap `select_move` never proposes a grow, so there is no at-cap grow bar. A (Flat) HINTS sweep that starts at the cap can still draw a grow after an earlier prune in the same sweep frees a node. That grow was drawn on a tree below the cap and is counted there.

In [ ]:
mcmc_cap_table, mcmc_cap_counts = rv.cap_acceptance_table(rv.by_proposal(mcmc_results))
smc_cap_table, smc_cap_counts = rv.cap_acceptance_table(rv.by_proposal(smc_results))

rv.plot_cap_fates(mcmc_cap_counts, "MCMC", footnote=rv.CAP_NOTE, ncols=2)
rv.plot_cap_fates(smc_cap_counts, "SMC", footnote=rv.SMC_CAP_NOTE, ncols=2)
display(mcmc_cap_table)
display(smc_cap_table)

In [ ]:
rv.plot_cap_fates_by_move(mcmc_cap_counts, "MCMC", footnote=rv.CAP_NOTE)
rv.plot_cap_fates_by_move(smc_cap_counts, "SMC", footnote=rv.SMC_CAP_NOTE)

## Proposal outcome traces
What each proposal came to, per iteration (MCMC) or step (SMC), stacked to 100%. One call per chain or particle, where a call is a whole sweep for (Flat) HINTS. A second figure follows every move drawn inside the HINTS sweeps, in the same colours.

In [ ]:
rv.plot_outcome_traces(mcmc_moves, "MCMC", ncols=2)
rv.plot_outcome_traces(smc_moves, "SMC", ncols=2, footnote=rv.SMC_TRACE_NOTE)